In [ ]:
# --- CÉLULA MESTRA: SETUP UNIVERSAL + IMPORTS ---
import os

# --- 1. CONFIGURAÇÃO DO GITHUB ---
REPO_URL = "https://github.com/ROMAUSKI/tcc-analise-sentimento.git"
NOME_REPO = "tcc-analise-sentimento"

print("--- Configurando Ambiente (Baixando dados do GitHub) ---")

# A. [IMPORTANTE] Volta para a pasta raiz do Colab antes de verificar
# Isso impede que, se você rodar a célula 2x, ele tente criar pastas uma dentro da outra
try:
    os.chdir("/content")
except:
    pass # Se não estiver no Colab, ignora

# B. Clona o repositório (SÓ SE ELE NÃO EXISTIR)
if not os.path.exists(NOME_REPO):
    print(f"📥 Clonando arquivos de: {REPO_URL}")
    !git clone $REPO_URL
else:
    print("✅ Repositório já baixado. Pulando clonagem.")

# C. Entra na pasta do projeto
try:
    if NOME_REPO not in os.getcwd():
        os.chdir(NOME_REPO)
        print(f"📍 Entrando na pasta do projeto: {os.getcwd()}")
except Exception as e:
    print(f"⚠️ Aviso de navegação: {e}")

# D. Entra na pasta 'dados/brutos' (onde estão os CSVs brutos)
try:
    if not os.getcwd().endswith('brutos'):
        os.chdir('dados/brutos')
        print(f"📂 Diretório de trabalho definido para: {os.getcwd()}")
except Exception as e:
    print("❌ Erro: Pasta 'dados/brutos' não encontrada. Verifique seu GitHub.")

# E. Garante que a pasta resultados existe
if not os.path.exists('../../resultados'):
    os.makedirs('../../resultados')
    print("📁 Pasta '../../resultados' verificada.")

# --- 2. IMPORTAÇÃO DE BIBLIOTECAS (SEU KIT DE FERRAMENTAS) ---
import pandas as pd
import numpy as np
import re
import glob
import warnings

# Gráficos
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn (Preparação e Modelos)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Métricas e Validação
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)
from sklearn.model_selection import cross_val_score, learning_curve
from sklearn.pipeline import Pipeline

# Configurações Visuais
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

print("\n🚀 Ambiente pronto! O notebook está conectado ao GitHub.")

In [95]:
#Versões Atuais das Bibliotecas Instaladas
!pip freeze | grep -E "pandas|scikit-learn|matplotlib|seaborn|nltk|wordcloud"

geopandas==1.1.1
matplotlib==3.10.0
matplotlib-inline==0.2.1
matplotlib-venn==1.1.2
nltk==3.9.1
pandas==2.2.2
pandas-datareader==0.10.0
pandas-gbq==0.30.0
pandas-stubs==2.2.2.240909
scikit-learn==1.6.1
seaborn==0.13.2
sklearn-pandas==2.2.0
wordcloud==1.9.4


In [ ]:
# Caminho dos dados
%cd "/content/tcc-analise-sentimento/dados/brutos"

# Confirme que você está no lugar certo (deve listar seus 9 CSVs + metadata)
!ls

In [ ]:
#UNIFICAÇÃO DOS 9 ARQUIVOS

print("--- Iniciando script de unificação ---")

# 1. Encontrar todos os arquivos CSV
try:
    arquivos_csv = glob.glob('*.csv')
    # Remove arquivos que não são dados brutos
    for arq_ignorar in ['metadata.csv', 'dataset_completo.csv', 'synthetic_dataset.csv']:
        if arq_ignorar in arquivos_csv:
            arquivos_csv.remove(arq_ignorar)

except ValueError:
    print("Aviso: Falha ao remover arquivos, processando todos os CSVs.")

print(f"Arquivos encontrados para unificação: {arquivos_csv}")

lista_dataframes = []

# 2. Loop para ler cada arquivo
for arquivo in arquivos_csv:
    try:
        # Leitura forçando a primeira coluna (correção do bug do GPT)
        df_temp = pd.read_csv(
            arquivo,
            header=None,
            names=['frase'],
            usecols=[0]
        )

        # Limpeza básica (nulos e vazios)
        df_temp = df_temp.dropna(subset=['frase'])
        df_temp['frase'] = df_temp['frase'].astype(str)
        df_temp = df_temp[df_temp['frase'].str.strip() != ""]

        # Adiciona metadados
        nome_arquivo = arquivo.split('.')[0]

        if 'positive' in nome_arquivo: df_temp['classe'] = 'Positiva'
        elif 'negative' in nome_arquivo: df_temp['classe'] = 'Negativa'
        elif 'neutral' in nome_arquivo: df_temp['classe'] = 'Neutra'
        else: df_temp['classe'] = 'Desconhecida'

        if '_gemini_' in nome_arquivo: df_temp['fonte'] = 'Gemini'
        elif '_gpt_' in nome_arquivo: df_temp['fonte'] = 'ChatGPT'
        elif '_claude_' in nome_arquivo: df_temp['fonte'] = 'Claude'
        else: df_temp['fonte'] = 'Desconhecida'

        lista_dataframes.append(df_temp)

    except Exception as e:
        print(f"Erro ao ler o arquivo {arquivo}: {e}")

# 3. Combinar tudo
df_final = pd.concat(lista_dataframes, ignore_index=True)

# 4. Embaralhar (COM SEED FIXA)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Salvar na pasta processado
df_final.to_csv('../processado/dataset_completo.csv', index=False)

# 6. Mostrar resumo
print("\n--- Processamento Concluído! ---")
print(f"Total de frases no dataset final: {len(df_final)}")
print("\nDistribuição por Classe:")
print(df_final['classe'].value_counts())

In [ ]:
#PRÉ PROCESSAMENTO (LIMPEZA)

print("--- Iniciando Etapa de Pré-processamento ---")

# 1. Carregar o dataset unificado que acabamos de criar
try:
    df = pd.read_csv('../processado/dataset_completo.csv')
    print(f"Dataset 'dataset_completo.csv' carregado. Total de {len(df)} linhas.")
except FileNotFoundError:
    print("ERRO: Arquivo 'dataset_completo.csv' não encontrado!")

# 2. Verificação inicial (pré-limpeza)
print(f"Frases duplicadas encontradas (antes da limpeza): {df.duplicated(subset=['frase']).sum()}")

# 3. Função de Limpeza de Texto
def limpar_texto(texto):
    # Converte tudo para minúsculas
    texto = texto.lower()

    # Remove pontuação, números e caracteres especiais
    # Mantém apenas letras, espaços e acentos
    texto = re.sub(r'[^a-zÀ-ÿ\s]', '', texto)

    # Remove espaços extras (ex: "filme   bom" vira "filme bom")
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto

# 4. Aplicar a limpeza na coluna 'frase'
print("Aplicando limpeza (minúsculas, remoção de pontuação/números)...")
df['frase_limpa'] = df['frase'].apply(limpar_texto)

# 5. Verificação pós-limpeza
duplicatas_pos_limpeza = df.duplicated(subset=['frase_limpa']).sum()
print(f"Frases duplicadas encontradas (depois da limpeza): {duplicatas_pos_limpeza}")

# Remove efetivamente as duplicatas
df = df.drop_duplicates(subset=['frase_limpa']).reset_index(drop=True)

# 6. Salvar o dataset final e limpo na pasta processado
df_final_limpo = df[['frase_limpa', 'classe', 'fonte', 'frase']]
df_final_limpo.to_csv('../processado/synthetic_dataset.csv', index=False)

print("\n--- Pré-processamento Concluído! ---")
print(f"Dataset limpo salvo em 'dados/processado/synthetic_dataset.csv'.")
print(f"Total de frases únicas no dataset final: {len(df_final_limpo)}")
print("\n10 primeiras linhas do dataset limpo:")
print(df_final_limpo.rename(columns={'frase': 'frase_original'}).head(10))

In [ ]:
print("--- Iniciando Etapa de Análise Estatística (com Violin Plot) ---")

# 1. Carregar o dataset limpo
try:
    df = pd.read_csv('../processado/synthetic_dataset.csv')
    print(f"Dataset 'synthetic_dataset.csv' carregado. Total de {len(df)} frases.")
except FileNotFoundError:
    print("ERRO: Arquivo 'synthetic_dataset.csv' não encontrado! Rode a célula anterior primeiro.")
    raise

# 2. Calcular estatísticas de texto
df['num_palavras'] = df['frase_limpa'].apply(lambda x: len(str(x).split()))

# --- EXIBIÇÃO DAS ESTATÍSTICAS (TEXTO) ---
print("\n--- ESTATÍSTICAS GERAIS DO DATASET ---")
print(f"Número total de frases (amostras): {len(df)}")
print("\n--- Distribuição por Classe ---")
print(df['classe'].value_counts())
print("\n--- Distribuição por Fonte (IA) ---")
print(df['fonte'].value_counts())
print("\n--- Estatísticas do Comprimento das Frases (Nº de Palavras) ---")
print(f"Média de palavras por frase: {df['num_palavras'].mean():.2f}")
print(f"Mediana de palavras por frase: {df['num_palavras'].median()}")
print(f"Frase mais curta (palavras): {df['num_palavras'].min()}")
print(f"Frase mais longa (palavras): {df['num_palavras'].max()}")

# --- GERAÇÃO DE GRÁFICOS (AVANÇADOS) ---

# Define o caminho para salvar
pasta_resultados = "../../resultados/"
print("\nSalvando gráficos na pasta 'resultados/'...")

# --- Gráfico 1 (O Combinado): Barras Agrupadas ---
plt.figure(figsize=(10, 6))
sns.countplot(
    data=df,
    x='classe',
    hue='fonte',
    order=['Positiva', 'Neutra', 'Negativa'],
    hue_order=['Gemini', 'ChatGPT', 'Claude']
)
plt.title('Distribuição de Frases por Classe e Fonte de IA')
plt.xlabel('Classe')
plt.ylabel('Contagem')
plt.legend(title='Fonte (IA)')
plt.savefig(f"{pasta_resultados}distribuicao_classe_por_fonte.png")
print(f"Gráfico 'distribuicao_classe_por_fonte.png' salvo.")

# --- Gráfico 2 (O "Mais Legal"): VIOLIN PLOT ---
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df,
    x='fonte',
    y='num_palavras',
    order=['Gemini', 'ChatGPT', 'Claude'],
    inner='quartile'
)
plt.title('Distribuição do Comprimento das Frases por Fonte de IA')
plt.xlabel('Fonte (IA)')
plt.ylabel('Número de Palavras')
plt.savefig(f"{pasta_resultados}violino_comprimento_por_fonte.png")
print(f"Gráfico 'violino_comprimento_por_fonte.png' salvo.")

# --- Gráfico 3 (O Útil): Histograma Geral ---
plt.figure(figsize=(10, 6))
sns.histplot(df['num_palavras'], bins=20, kde=True)
plt.title('Distribuição Geral do Comprimento das Frases')
plt.xlabel('Número de Palavras')
plt.ylabel('Frequência')
plt.savefig(f"{pasta_resultados}distribuicao_comprimento_frases.png")
print(f"Gráfico 'distribuicao_comprimento_frases.png' salvo.")

print("\n--- Análise Estatística Concluída! ---")

In [ ]:
print("--- Iniciando Etapa 3: Preparação dos Modelos Baseline ---")

# --- [X] Carregar o synthetic_dataset.csv ---
try:
    df = pd.read_csv('../processado/synthetic_dataset.csv')
    print(f"Dataset 'synthetic_dataset.csv' carregado com {len(df)} linhas.")
except FileNotFoundError:
    print("ERRO: 'synthetic_dataset.csv' não encontrado! Rode as células anteriores.")
    raise
except Exception as e:
    print(f"Erro ao carregar o CSV: {e}")
    raise

# Garantir que não há nulos na coluna 'frase_limpa' (embora já tenhamos tratado)
df = df.dropna(subset=['frase_limpa'])

# 1. Definir X (features) e y (target)
X = df['frase_limpa']
y = df['classe']

# 2. Definir nossa "semente" (seed) para reprodutibilidade
# Isso é VITAL para o seu TCC. Garante que os resultados sejam sempre os mesmos.
RANDOM_SEED = 42

# --- [X] Dividir os dados em treino (80%) e teste (20%) ---
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y
)

print("\n--- Divisão Treino/Teste Concluída ---")
print(f"Amostras de Treino (X_train): {len(X_train)}")
print(f"Amostras de Teste (X_test):   {len(X_test)}")
print(f"Rótulos de Treino (y_train): {len(y_train)}")
print(f"Rótulos de Teste (y_test):   {len(y_test)}")

# --- [X] Implementar a vetorização com TF-IDF ---
print("\n--- Iniciando Vetorização TF-IDF ---")

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("Vetorização TF-IDF Concluída.")
print(f"Shape da Matriz TF-IDF de Treino: {X_train_tfidf.shape}")
print(f"Shape da Matriz TF-IDF de Teste:  {X_test_tfidf.shape}")
print(f"Número total de features (palavras únicas): {X_train_tfidf.shape[1]}")

In [ ]:
print("--- Etapa 3.2: Treinamento - Regressão Logística (Análise F1) ---")

# 1. Inicializar e Treinar o modelo
model_lr = LogisticRegression(random_state=RANDOM_SEED)
print("Treinando o modelo de Regressão Logística...")
model_lr.fit(X_train_tfidf, y_train)
print("Treinamento concluído.")

# 2. Fazer previsões
y_pred_lr = model_lr.predict(X_test_tfidf)

# 3. Gerar Relatório
print("\n--- Relatório de Classificação (Completo) ---")
print(classification_report(y_test, y_pred_lr))

# 4. Gerar o Relatório como um Dicionário
report_dict = classification_report(y_test, y_pred_lr, output_dict=True)

# 5. Converter pra um DataFrame do Pandas
df_report = pd.DataFrame(report_dict).transpose()

# 6. Pegar só o F1-Score das 3 classes principais
f1_scores = df_report.loc[['Negativa', 'Neutra', 'Positiva'], 'f1-score']

# 7. Imprimir o F1-Score "Ponderado"
f1_final = report_dict['weighted avg']['f1-score']
print(f"\n--- Veredito (O que importa pro TCC) ---")
print(f"O F1-Score final (ponderado) do modelo foi: {f1_final:.4f} (ou {f1_final*100:.2f}%)")

# 8. Plotar o Gráfico de F1-Score por Classe
pasta_resultados = "../../resultados/"
plt.figure(figsize=(8, 5))
sns.barplot(
    x=f1_scores.index,
    y=f1_scores.values,
    palette='viridis',
    hue=f1_scores.index,
    legend=False
)
plt.title('F1-Score por Classe - Regressão Logística')
plt.xlabel('Classe')
plt.ylabel('F1-Score')
plt.ylim(0, 1.0)
plt.savefig(f"{pasta_resultados}f1_score_por_classe_lr.png")
print(f"Gráfico 'f1_score_por_classe_lr.png' salvo em {pasta_resultados}")
plt.show()

# Gerar Matriz de Confusão
print("\nGerando Matriz de Confusão...")
cm_lr = confusion_matrix(y_test, y_pred_lr)
labels = sorted(y.unique())

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_lr,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=labels,
    yticklabels=labels
)
plt.title('Matriz de Confusão - Regressão Logística')
plt.xlabel('Classe Prevista')
plt.ylabel('Classe Verdadeira')
plt.savefig(f"{pasta_resultados}matriz_confusao_lr.png")
print(f"Matriz 'matriz_confusao_lr.png' salva em {pasta_resultados}")
plt.show()

# Armazena os resultados pro nosso dicionário
try:
    resultados_modelos
except NameError:
    resultados_modelos = {}

resultados_modelos['Regressão Logística'] = {
    'Acurácia': report_dict['accuracy'],
    'Precisão': report_dict['weighted avg']['precision'],
    'Recall': report_dict['weighted avg']['recall'],
    'F1-Score': f1_final
}

print("\nResultados da Regressão Logística salvos.")

In [ ]:
print("--- Etapa 3.3: Treinamento - Naive Bayes (Multinomial) ---")

# 1. Inicializar o modelo
model_nb = MultinomialNB()

# 2. Treinar o modelo
print("Treinando o modelo Naive Bayes...")
model_nb.fit(X_train_tfidf, y_train)
print("Treinamento concluído.")

# 3. Fazer previsões
y_pred_nb = model_nb.predict(X_test_tfidf)

# 4. Gerar Relatório
print("\n--- Relatório de Classificação (Naive Bayes) ---")
print(classification_report(y_test, y_pred_nb))

# 5. Gerar o Relatório como um Dicionário
report_dict_nb = classification_report(y_test, y_pred_nb, output_dict=True)
df_report_nb = pd.DataFrame(report_dict_nb).transpose()
f1_scores_nb = df_report_nb.loc[['Negativa', 'Neutra', 'Positiva'], 'f1-score']

f1_final_nb = report_dict_nb['weighted avg']['f1-score']
print(f"\n--- Veredito (O que importa pro TCC) ---")
print(f"O F1-Score final (ponderado) do modelo foi: {f1_final_nb:.4f} (ou {f1_final_nb*100:.2f}%)")

# 6. Plotar o Gráfico de F1-Score por Classe
pasta_resultados = "../../resultados/"
plt.figure(figsize=(8, 5))
sns.barplot(
    x=f1_scores_nb.index,
    y=f1_scores_nb.values,
    palette='viridis',
    hue=f1_scores_nb.index,
    legend=False
)
plt.title('F1-Score por Classe - Naive Bayes')
plt.xlabel('Classe')
plt.ylabel('F1-Score')
plt.ylim(0, 1.0)
plt.savefig(f"{pasta_resultados}f1_score_por_classe_nb.png")
print(f"Gráfico 'f1_score_por_classe_nb.png' salvo em {pasta_resultados}")
plt.show()

# 7. Gerar Matriz de Confusão
print("\nGerando Matriz de Confusão...")
cm_nb = confusion_matrix(y_test, y_pred_nb)
labels = sorted(y.unique())

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_nb,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=labels,
    yticklabels=labels
)
plt.title('Matriz de Confusão - Naive Bayes')
plt.xlabel('Classe Prevista')
plt.ylabel('Classe Verdadeira')
plt.savefig(f"{pasta_resultados}matriz_confusao_nb.png")
print(f"Matriz 'matriz_confusao_nb.png' salva em {pasta_resultados}")
plt.show()

# 8. Armazena os resultados no nosso dicionário
resultados_modelos['Naive Bayes'] = {
    'Acurácia': report_dict_nb['accuracy'],
    'Precisão': report_dict_nb['weighted avg']['precision'],
    'Recall': report_dict_nb['weighted avg']['recall'],
    'F1-Score': f1_final_nb
}

print("\nResultados do Naive Bayes salvos.")

In [ ]:
print("--- [X] Relatório Final: Comparativo dos Modelos Baseline ---\n")

# 1. PRINTAR NA TELA
for nome_modelo, metricas in resultados_modelos.items():
    print(f"--- {nome_modelo} ---")
    acc = f"{metricas['Acurácia'] * 100:.2f}%"
    pre = f"{metricas['Precisão'] * 100:.2f}%"
    rec = f"{metricas['Recall'] * 100:.2f}%"
    f1  = f"{metricas['F1-Score'] * 100:.2f}%\n"
    print(f" Acurácia: {acc}")
    print(f" Precisão: {pre}")
    print(f" Recall:   {rec}")
    print(f" F1-Score: {f1}")
    print("--------------------------------------\n")

# 2. SALVAR NO ARQUIVO
print("--- Salvando resultados em arquivo... ---")

df_resultados = pd.DataFrame.from_dict(resultados_modelos, orient='index')
df_resultados = df_resultados.round(4)

caminho_arquivo = "../../resultados/baseline_metrics.csv"

try:
    df_resultados.to_csv(caminho_arquivo)
    print(f"✅ Sucesso! Arquivo salvo em: {caminho_arquivo}")
    print("\nConteúdo salvo no CSV:")
    print(df_resultados)
except Exception as e:
    print(f"❌ Erro ao salvar o arquivo: {e}")

In [ ]:
print("--- Etapa 4.1: Validação Cruzada (k-fold) ---")

k_folds = 10

pipeline_nb = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

print(f"Rodando Validação Cruzada (k={k_folds}) para Naive Bayes...")
print("Isso pode levar alguns segundos...")

scores = cross_val_score(
    pipeline_nb,
    X,
    y,
    cv=k_folds,
    scoring='f1_weighted'
)

print("\n--- Resultados da Estabilidade ---")
print(f"F1-Scores das {k_folds} rodadas: {scores}")
print(f"Média do F1-Score: {scores.mean():.4f} (ou {scores.mean()*100:.2f}%)")
print(f"Desvio Padrão: +/- {scores.std():.4f}")

if scores.std() < 0.05:
    print("\n>> CONCLUSÃO: O modelo é ESTÁVEL (baixo desvio padrão).")
else:
    print("\n>> CONCLUSÃO: O modelo é INSTÁVEL (alto desvio padrão).")

# Atualizar o arquivo de métricas
try:
    resultados_modelos['NB (Validação Cruzada)'] = {
        'Acurácia': np.nan,
        'Precisão': np.nan,
        'Recall': np.nan,
        'F1-Score': scores.mean()
    }

    df_resultados = pd.DataFrame.from_dict(resultados_modelos, orient='index').round(4)
    df_resultados.to_csv("../../resultados/baseline_metrics.csv")

    print("\n✅ Sucesso! O arquivo 'baseline_metrics.csv' foi atualizado com a Validação Cruzada.")
    print(df_resultados.tail(1))

except NameError:
    print("⚠️ Aviso: Dicionário 'resultados_modelos' não encontrado. O CSV não foi atualizado.")

## Texto para o artigo

### Metodologia — Construção do Dataset

O dataset utilizado neste trabalho foi construído inteiramente a partir de dados sintéticos gerados por três grandes modelos de linguagem (LLMs): ChatGPT da OpenAI, Claude da Anthropic e Gemini do Google. A escolha por utilizar múltiplos modelos teve como objetivo reduzir o viés de geração de uma única fonte, já que cada LLM possui características próprias de vocabulário, estrutura de frases e estilo de escrita.

Para cada modelo, foram solicitadas 200 frases em português brasileiro para cada uma das três classes de sentimento — positiva, negativa e neutra — no domínio de críticas de cinema e séries de TV. A geração foi feita manualmente por meio das interfaces web dos respectivos modelos, com prompts padronizados descrevendo o tipo de frase desejada. No total, foram gerados nove arquivos CSV (3 modelos × 3 classes), que foram unificados em um dataset único contendo a frase original, a classe de sentimento e a identificação da fonte geradora.

Na etapa de pré-processamento, as frases foram convertidas para minúsculas, tiveram pontuação, números e caracteres especiais removidos, e espaços extras foram normalizados. Após a eliminação de 2 frases duplicadas detectadas pela limpeza, o dataset final resultou em 1798 amostras únicas, com distribuição praticamente equilibrada entre as classes (600 positivas, 600 neutras e 598 negativas).

### Metodologia — Modelos e Avaliação

Dois classificadores foram selecionados como modelos baseline: Naive Bayes Multinomial e Regressão Logística. Ambos são amplamente utilizados na literatura de classificação de textos por sua simplicidade, interpretabilidade e bom desempenho em tarefas com representação esparsa como o TF-IDF.

A vetorização do texto foi realizada com TF-IDF (Term Frequency — Inverse Document Frequency), que transforma as frases em vetores numéricos ponderando a importância de cada termo no corpus. O dataset foi dividido em 80% para treino e 20% para teste, com estratificação por classe para manter a proporção original em ambas as partições. Uma semente fixa (random_state=42) foi utilizada em todas as etapas para garantir reprodutibilidade completa dos experimentos.

As métricas de avaliação adotadas foram acurácia, precisão, recall e F1-Score ponderado (weighted), escolhido por considerar o suporte de cada classe no cálculo da média.

### Resultados — Modelos Baseline

O Naive Bayes Multinomial obteve acurácia de 90,28% e F1-Score ponderado de 90,30%, superando a Regressão Logística que alcançou 85,83% de acurácia e 85,84% de F1-Score. Ambos os modelos apresentaram desempenho equilibrado entre as três classes, sem viés significativo para nenhuma delas, o que era esperado dada a distribuição balanceada do dataset.

A superioridade do Naive Bayes nesse cenário pode ser explicada pela natureza do TF-IDF: como a vetorização assume independência entre as features (palavras), essa premissa se alinha diretamente com a suposição fundamental do Naive Bayes, que assume independência condicional entre os atributos dada a classe. A Regressão Logística, por buscar fronteiras de decisão lineares no espaço de features, pode ser mais sensível a ruídos e sobreposições no vocabulário entre classes.

A análise da matriz de confusão do Naive Bayes revelou que a maior concentração de erros ocorre entre as classes Positiva e Negativa, sugerindo que parte das frases geradas pelos LLMs compartilham vocabulário ambíguo que dificulta a discriminação por modelos baseados em bag-of-words. Esse ponto é aprofundado na análise de erros do Notebook 02.

---

### Referências para consultar antes de escrever o artigo

Sobre análise de sentimentos e classificação de texto:
- Liu, B. (2012). *Sentiment Analysis and Opinion Mining*. Morgan & Claypool Publishers. — referência clássica da área, define conceitos fundamentais
- Pang, B., & Lee, L. (2008). *Opinion Mining and Sentiment Analysis*. Foundations and Trends in Information Retrieval, 2(1-2), 1-135. — survey seminal, contextualiza técnicas tradicionais

Sobre os modelos utilizados (Naive Bayes e Regressão Logística para texto):
- Jurafsky, D., & Martin, J. H. (2024). *Speech and Language Processing* (3ª ed., rascunho). Cap. 4 (Naive Bayes) e Cap. 5 (Logistic Regression). Disponível em: https://web.stanford.edu/~jurafsky/slp3/ — explica os dois modelos no contexto de NLP
- Manning, C. D., Raghavan, P., & Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press. Cap. 13 (Text Classification and Naive Bayes). — fundamentação do TF-IDF e Naive Bayes

Sobre dados sintéticos gerados por LLMs:
- Veselovsky, V., Ribeiro, M. H., & West, R. (2023). *Generating Faithful Synthetic Data with Large Language Models: A Case Study in Computational Social Science*. arXiv:2305.15041. — discute qualidade e vieses de dados gerados por LLMs
- Li, Y. et al. (2023). *Synthetic Data Generation with Large Language Models for Text Classification: Potential and Limitations*. EMNLP 2023. — compara dados sintéticos vs reais para classificação de texto, bem alinhado com o TCC
- Møller, A. G. et al. (2024). *Is a prompt and a few samples all you need? Using GPT-4 for data augmentation in low-resource classification tasks*. arXiv:2304.13861.

Sobre avaliação e validação de modelos:
- Scikit-learn documentation: Cross-validation, Learning Curves. https://scikit-learn.org/stable/modules/cross_validation.html — referência técnica dos métodos usados
- Raschka, S. (2018). *Model Evaluation, Model Selection, and Algorithm Selection in Machine Learning*. arXiv:1811.12808. — bom pra fundamentar as escolhas de avaliação

Sobre análise de sentimentos em português:
- Santos, W. R. et al. (2022). *Multilingual transformers for sentiment analysis of tweets: the case of Brazilian Portuguese*. — contextualiza NLP em PT-BR
- Hartmann, N. et al. (2014). *A Large Corpus of Product Reviews in Portuguese: Tackling Out-Of-Vocabulary Words*. LREC 2014. — um dos poucos datasets de sentimento em PT-BR